# Nimbus Build 1 — Lakebase validation

Executed evidence for `projects/nimbus-growth-ops/branches/main`, database `nimbus`. The SQL executed for this export is preserved verbatim in `lakebase_validation.sql`.

In [1]:
import json
from pathlib import Path

evidence = json.loads(Path('lakebase_validation_result.json').read_text())
tables = {t['table_name']: t for t in evidence['operational_tables']}
constraints = evidence['constraints']
forecast = evidence['forecast']
write = evidence['application_role_test']
synced = evidence['synced_table_test']
print(f"Operational tables: decision_forecasts={tables['decision_forecasts']['row_count']}, feature_decisions_app={tables['feature_decisions_app']['row_count']}")
print(f"Keys: {sum(c['constraint_type'] == 'PRIMARY KEY' for c in constraints)} PRIMARY KEY, {sum(c['constraint_type'] == 'FOREIGN KEY' for c in constraints)} FOREIGN KEY")
print(f"Replica identity: decision_forecasts={tables['decision_forecasts']['replica_identity']}, feature_decisions_app={tables['feature_decisions_app']['replica_identity']}")
print(f"Forecast: {forecast['segment_id']}, lift={forecast['forecast_conversion_lift']}, at_risk_usd={forecast['forecast_conversion_at_risk_usd']}")
print(f"Application role INSERT: {write['operational_insert']}; transaction {write['transaction']}")
print(f"Synced table SELECT: {synced['select']} ({synced['sample_segment_id']})")
print(f"Synced table INSERT: {synced['insert'].replace(':', ' (', 1)})" + (')' if ':' in synced['insert'] else ''))

Operational tables: decision_forecasts=1, feature_decisions_app=1
Keys: 2 PRIMARY KEY, 1 FOREIGN KEY
Replica identity: decision_forecasts=FULL, feature_decisions_app=FULL
Forecast: SEG-0000214, lift=0.0225, at_risk_usd=524160
Application role INSERT: SUCCEEDED; transaction ROLLED BACK
Synced table SELECT: SUCCEEDED (SEG-0000001)
Synced table INSERT: DENIED (permission denied for table segment_positions)


## Verification conclusion

Both operational tables exist with rows, primary keys, the required relationship, and `REPLICA IDENTITY FULL`. The application role can write only to the operational surface; its proof write was rolled back. The synced serving table is readable and rejects INSERT.